# Chunking
Divide extracted text into smaller chunks, each focused on a single topic.

Alternatives:
Proprietary - Gemini
Open Source - Gemma3-4b

Run this in terminal to start llama server:
```bash
./llama.cpp/llama-server.exe \
  --model models/gemma-3-4b-it-Q4_K_M.gguf \
  --ctx-size 16384 \
  --n-gpu-layers 999 \
  --threads 4 \
  --batch-size 4096 \
  --ubatch-size 1024 \
  --flash-attn auto \
  --mlock \
  --port 36912
```
```bash
  ./llama.cpp/llama-server.exe \
  --model models/gemma-3-4b-it-Q8_0.gguf \
  --ctx-size 16384 \
  --n-gpu-layers 999 \
  --threads 4 \
  --batch-size 2048 \
  --ubatch-size 512 \
  --flash-attn auto \
  --mlock \
  --port 36912
  ```

In [1]:
system_prompt = """\
[
{"text_chunk": "REPLACE_THIS_WITH_FIRST_REAL_CHUNK"}
]

The above is the EXACT format you must output. Your response must begin with [ on the very first line.

---

You are a text segmentation engine for automotive user manuals.

## YOUR ONLY JOB

Split the input text into AS MANY CHUNKS AS THERE ARE DISTINCT TOPICS.
You are a SPLITTER, not a writer. You are not a merger. You are not a summarizer.

A section with 10 different topics must produce 10 chunks.
A section with 20 different topics must produce 20 chunks.
Never combine different topics into one chunk.

ABSOLUTE RULES — violating any of these makes your output useless:

RULE 1 — YOUR RESPONSE MUST START WITH [
The very first character you output must be [
Do NOT write any explanation, introduction, or prose before the JSON array.
Begin your response with [ immediately.

RULE 2 — COPY TEXT VERBATIM
Every character in every chunk must be copied letter-for-letter from the input.
Do NOT change a single word, punctuation mark, or line break.
Do NOT paraphrase. Do NOT summarize. Do NOT rewrite. Do NOT improve.

RULE 3 — ZERO OMISSIONS
Every single word from the input must appear in exactly one chunk.
Nothing may be skipped, shortened, or left out.
If you finish and the chunks do not contain 100% of the input text, you have failed.

RULE 4 — NO ADDITIONS
Do not add any text that is not in the input.
Do not add headings, labels, summaries, or explanations.

RULE 5 — COMPLETE JSON ONLY
Output a single JSON array. Nothing before it. Nothing after it.
The very first character of your response must be [ and the last must be ].
Do NOT wrap your output in markdown code fences like ```json ... ```

---

## WHAT A CHUNK IS

A chunk = one logically complete topic from the manual.

IMPORTANT: Most input sections will contain MANY different topics.
You must identify each topic separately and create a separate chunk for each one.

One topic means one of these:
- A named vehicle feature and all its description paragraphs
- A complete operating procedure with all its numbered steps
- A warranty clause (qualification, term, obligation, exclusion — each is its own chunk)
- A group of related warning / caution / notice blocks for the same feature
- A maintenance procedure from start to finish
- A troubleshooting entry (symptom + cause + fix)
- An administrative section (foreword, how to download manual, etc.)

A topic ends when ANY of these happen:
- A new heading appears (##, ###, or a bold standalone title line)
- The subject shifts from one feature to a different unrelated feature
- A new numbered clause begins (e.g. warranty clause 1, clause 2, clause 3)

---

## GROUPING RULES

KEEP TOGETHER — never split these across chunks:
- A heading and the paragraphs directly below it that describe THE SAME topic
- A numbered procedure: all steps stay in one chunk
- A WARNING / CAUTION / NOTICE / NOTE block stays with the instruction it belongs to
- A bullet list stays intact in one chunk if all items are about the same topic
- A figure description stays with the section it illustrates

START A NEW CHUNK when:
- A new heading appears
- The topic clearly shifts to a different vehicle system or feature
- A new numbered clause or section begins

DO NOT MERGE different topics into one chunk even if they appear close together.
Each distinct topic gets its own chunk, always.

---

## OUTPUT FORMAT

Output a JSON array of objects. Each object has exactly one key: "text_chunk".
The value is the verbatim text of that segment as a plain string.

To include a newline inside a JSON string value, write it as \\n (JSON escape).
Do not use actual line breaks inside a string value — that produces invalid JSON.
Do NOT wrap your output in markdown fences.

[
  {"text_chunk": "First heading\\n\\nFirst paragraph text."},
  {"text_chunk": "Second heading\\n\\nSecond paragraph text.\\n\\nWARNING\\nWarning text."},
  {"text_chunk": "Third heading\\n\\nStep 1: Do this.\\nStep 2: Do that."}
]

---

## EXAMPLE

INPUT:

## FOREWORD

Please read this manual carefully before operating your vehicle.
Download this manual from the Maruti Suzuki mobile application.

## WARNING/ CAUTION/ NOTICE

WARNING
Indicates a potential hazard that could result in death or serious injury.

CAUTION
Indicates a potential hazard that could result in minor or moderate injury.

NOTICE
Indicates a potential hazard that could result in vehicle damage.

NOTE:
Indicates special information to make maintenance easier or instructions clearer.

## MODIFICATION WARNING

WARNING
Do not modify your vehicle. Modification could adversely affect safety, handling,
performance, or durability and may violate governmental regulations.

## WARRANTY POLICY

Maruti Suzuki warrants that each new vehicle will be free from defects in material
and workmanship at the time of manufacture.

## (1) Qualification

To qualify for this warranty the vehicle must be delivered by a Maruti Suzuki
authorized dealer and serviced by a Maruti Suzuki authorized workshop.

## (2) Term

The term of this warranty shall be Thirty-Six (36) months or 1,00,000 kilometers
whichever occurs first from the date of invoice to the first owner.

CORRECT OUTPUT (6 chunks — one per topic):
[
  {"text_chunk": "## FOREWORD\\n\\nPlease read this manual carefully before operating your vehicle.\\nDownload this manual from the Maruti Suzuki mobile application."},
  {"text_chunk": "## WARNING/ CAUTION/ NOTICE\\n\\nWARNING\\nIndicates a potential hazard that could result in death or serious injury.\\n\\nCAUTION\\nIndicates a potential hazard that could result in minor or moderate injury.\\n\\nNOTICE\\nIndicates a potential hazard that could result in vehicle damage.\\n\\nNOTE:\\nIndicates special information to make maintenance easier or instructions clearer."},
  {"text_chunk": "## MODIFICATION WARNING\\n\\nWARNING\\nDo not modify your vehicle. Modification could adversely affect safety, handling,\\nperformance, or durability and may violate governmental regulations."},
  {"text_chunk": "## WARRANTY POLICY\\n\\nMaruti Suzuki warrants that each new vehicle will be free from defects in material\\nand workmanship at the time of manufacture."},
  {"text_chunk": "## (1) Qualification\\n\\nTo qualify for this warranty the vehicle must be delivered by a Maruti Suzuki\\nauthorized dealer and serviced by a Maruti Suzuki authorized workshop."},
  {"text_chunk": "## (2) Term\\n\\nThe term of this warranty shall be Thirty-Six (36) months or 1,00,000 kilometers\\nwhichever occurs first from the date of invoice to the first owner."}
]

WRONG OUTPUT — never do this:
[
  {"text_chunk": "## FOREWORD\\n\\nPlease read this manual carefully...## (2) Term\\n\\nThe term of this warranty..."}
]

The WRONG output merges all topics into one chunk.
Every ## heading in the input must produce a separate chunk in the output.

---

## SELF-CHECK BEFORE OUTPUTTING

1. Does my response start with [ ? (must be YES)
2. Does every ## heading in the input have its own chunk? (must be YES)
3. Does every numbered clause have its own chunk? (must be YES)
4. Do my chunks combined contain every word from the input? (must be YES)
5. Did I copy verbatim — zero words changed or invented? (must be YES)
6. Are newlines written as \\n inside strings? (must be YES)
7. Any markdown fence or text outside the array? (must be NO)
"""

In [2]:
user_prompt = """\
Split the input text below into chunks. One chunk per distinct topic.
Every ## heading must become a separate chunk.
Start your response with [ immediately.

INPUT TEXT:
"""

In [3]:
#Read Extracted Text
import os

file_path = "../data/extracted_text/granite_docling_output.txt"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

with open(file_path, "r", encoding="utf-8") as file:
    extracted_text = file.read()

char_count = len(extracted_text)
approx_tokens = char_count // 4  # fast estimate (replacing expensive tokenizer)

print("Characters in extracted text:", char_count)
print("Estimated tokens:", approx_tokens)

Characters in extracted text: 1945221
Estimated tokens: 486305


In [4]:
#Configuration
import json
import requests
from typing import List

LLAMA_SERVER_URL = "http://127.0.0.1:36912"
MAX_CONTEXT_SIZE = 16384

# Tighter input limit — accounts for estimation error (len/3.8 underestimates by ~15%)
SAFE_INPUT_LIMIT = 2500

# Output = input * 1.3 overhead, but cap total context budget
# 800 (system) + 2500 (input) + 4500 (output) = 7800 — well within 16384
MAX_OUTPUT_TOKENS = 4500

MIN_CHUNK_CHARS = 200
MAX_CHUNK_CHARS = 3500

def estimate_tokens(text: str) -> int:
    # Deliberately pessimistic — better to split more than to overflow
    return int(len(text) / 3.2)   # was 3.8, which underestimates consistently

def count_tokens(text: str) -> int:
    try:
        response = requests.post(
            url=f"{LLAMA_SERVER_URL}/tokenize",
            headers={"Content-Type": "application/json"},
            data=json.dumps({"content": text})
        )
        response.raise_for_status()
        return len(response.json().get("tokens", []))
    except requests.exceptions.RequestException as e:
        print("Tokenization error:", e)
        return int(len(text) / 3.2)

In [5]:
#Count Tokens Function
def count_tokens(text: str) -> int:
    """
    Count tokens using the llama.cpp tokenizer via the /tokenize endpoint.
    """

    try:
        response = requests.post(
            url=f"{LLAMA_SERVER_URL}/tokenize",
            headers={"Content-Type": "application/json"},
            data=json.dumps({
                "content": text
            })
        )

        response.raise_for_status()

        tokens = response.json().get("tokens", [])

        return len(tokens)

    except requests.exceptions.RequestException as e:
        print("Tokenization error:", e)
        return -1

In [6]:
#fast local estimation

def estimate_tokens(text: str) -> int:
    """
    Fast token estimation.

    Approximation:
    1 token ≈ 4 characters (works well for English text)

    This avoids expensive API calls to llama server.
    """

    return len(text) // 3.8

In [7]:
#Analysis

system_prompt_tokens = count_tokens(system_prompt)
user_prompt_tokens = count_tokens(user_prompt)
document_tokens = count_tokens(extracted_text)

esystem_prompt_tokens = estimate_tokens(system_prompt)
euser_prompt_tokens = estimate_tokens(user_prompt)
edocument_tokens = estimate_tokens(extracted_text)

total_tokens = system_prompt_tokens + user_prompt_tokens + document_tokens

print("\n--- TOKEN ANALYSIS ---")
print(f"Est. System prompt tokens : {esystem_prompt_tokens}")
print(f"Est. User prompt tokens   : {euser_prompt_tokens}")
print(f"Est. Document tokens      : {edocument_tokens}")
print("----------------------")
print(f"System prompt tokens : {system_prompt_tokens}")
print(f"User prompt tokens   : {user_prompt_tokens}")
print(f"Document tokens      : {document_tokens}")
print("----------------------")
print(f"Total tokens         : {total_tokens}")
print(f"Context limit        : {MAX_CONTEXT_SIZE}")

if total_tokens > MAX_CONTEXT_SIZE:
    print("\nWARNING: Prompt exceeds model context window.")
    print("The document must be split before sending it to the LLM.\n")
else:
    print("\nPrompt fits within the model context window.\n")


--- TOKEN ANALYSIS ---
Est. System prompt tokens : 1918.0
Est. User prompt tokens   : 44.0
Est. Document tokens      : 511900.0
----------------------
System prompt tokens : 1754
User prompt tokens   : 37
Document tokens      : 544243
----------------------
Total tokens         : 546034
Context limit        : 16384

The document must be split before sending it to the LLM.



In [8]:
#Stream Completion
import requests
import json

GEMMA_STOP_TOKENS = ["<eos>", "<end_of_turn>", "<|end|>"]


def stream_completion(
    system_prompt: str,
    extracted_text: str,
    user_prompt: str,
    timeout_connect: int = 10,
    timeout_stream: int = 180,
):
    """
    Use /v1/chat/completions instead of /v1/completions.
    Gemma is a chat-tuned model — it expects the chat template.
    Using raw /v1/completions bypasses the template and causes
    hallucination and format ignoring.
    """

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt + extracted_text
        },
        {
            # Assistant pre-fill — forces model to continue from [
            # instead of deciding its own output format.
            # This is the correct way to pre-fill in chat completions.
            "role": "assistant",
            "content": "["
        }
    ]

    data = {
        "model": "gemma",          # llama.cpp ignores this but field is required
        "messages": messages,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "temperature": 0,
        "seed": 42,
        "stream": True,
        "stop": GEMMA_STOP_TOKENS,
        "cache_prompt": True,
    }

    try:
        response = requests.post(
            url=f"{LLAMA_SERVER_URL}/v1/chat/completions",
            headers={"Content-Type": "application/json"},
            json=data,
            stream=True,
            timeout=(timeout_connect, timeout_stream),
        )
        response.raise_for_status()

        # Pre-fill means [ is already injected via assistant message.
        # Yield it first so the assembled response is valid JSON.
        yield "["

        for line in response.iter_lines():
            if not line:
                continue

            decoded = line.decode("utf-8")
            if not decoded.startswith("data: "):
                continue

            payload = decoded[6:]
            if payload.strip() == "[DONE]":
                break

            try:
                chunk = json.loads(payload)
                # chat/completions uses delta.content, not text
                token = chunk["choices"][0]["delta"].get("content", "")
                if token:
                    yield token
            except (json.JSONDecodeError, KeyError, IndexError):
                continue

    except requests.exceptions.ConnectError:
        raise RuntimeError(
            f"Cannot connect to llama.cpp server at {LLAMA_SERVER_URL}. "
            "Is the server running?"
        )
    except requests.exceptions.ReadTimeout:
        raise RuntimeError(
            f"Stream stalled — no token received within {timeout_stream}s."
        )
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"HTTP error during streaming: {e}")

In [9]:
#Old Splitter
# import re
# import json
# import os
# from typing import List

# SECTION_FILE = "../data/chunks/llm_sections.jsonl"


# # 🔧 NEW: simple cache to avoid repeated token calls on same text
# TOKEN_CACHE = {}


# def get_tokens(text: str) -> int:
#     if text in TOKEN_CACHE:
#         return TOKEN_CACHE[text]

#     tokens = count_tokens(text)
#     TOKEN_CACHE[text] = tokens
#     return tokens


# def split_large_section(section: str, max_tokens: int):
#     """
#     Fallback splitter for oversized markdown sections.
#     Splits by paragraphs so no chunk exceeds token limit.
#     """

#     paragraphs = section.split("\n\n")

#     chunks = []
#     current = ""

#     for para in paragraphs:

#         candidate = current + "\n\n" + para if current else para

#         # 🔧 UPDATED: use actual tokens
#         if get_tokens(candidate) > max_tokens:

#             if current:
#                 chunks.append(current)
#                 current = para
#             else:
#                 chunks.append(para)
#                 current = ""

#         else:
#             current = candidate

#     if current:
#         chunks.append(current)

#     return chunks


# def split_text_by_tokens(
#     text: str,
#     max_tokens: int = SAFE_INPUT_LIMIT,
#     overlap_sections: int = 1
# ) -> int:

#     print("\n===== SPLITTER START =====")

#     os.makedirs(os.path.dirname(SECTION_FILE), exist_ok=True)
#     open(SECTION_FILE, "w").close()

#     # Step 1 — Split by markdown headings
#     print("Detecting markdown sections...")

#     raw_sections = re.split(r"\n## ", text)

#     sections = []
#     for i, sec in enumerate(raw_sections):
#         if i == 0:
#             sections.append(sec)
#         else:
#             sections.append("## " + sec)

#     print("Total sections detected:", len(sections))

#     # Step 2 — Fix oversized sections
#     print("\nChecking for oversized sections...")

#     fixed_sections = []

#     for sec in sections:

#         tokens = get_tokens(sec)

#         if tokens <= max_tokens:
#             fixed_sections.append(sec)

#         else:
#             print(f"Oversized section detected (actual = {tokens}) → splitting")

#             sub_chunks = split_large_section(sec, max_tokens)
#             fixed_sections.extend(sub_chunks)

#     sections = fixed_sections

#     print("Sections after fixing oversized ones:", len(sections))

#     # Step 3 — Combine sections under token limit
#     print("\nCombining sections while respecting token limit...")

#     start_index = 0
#     chunk_counter = 0

#     while start_index < len(sections):

#         current_text = ""
#         current_sections = []

#         i = start_index

#         while i < len(sections):

#             candidate_text = current_text + "\n" + sections[i]

#             # 🔧 UPDATED: actual tokens ONLY
#             token_count = get_tokens(candidate_text)

#             if token_count > max_tokens:
#                 break

#             current_text = candidate_text
#             current_sections.append(sections[i])

#             i += 1

#         chunk_text = "\n".join(current_sections).strip()

#         if not chunk_text:
#             start_index += 1
#             continue

#         # SAVE TO DISK
#         with open(SECTION_FILE, "a", encoding="utf-8") as f:
#             json.dump({"section_text": chunk_text}, f)
#             f.write("\n")

#         chunk_counter += 1

#         # 🔧 UPDATED: actual tokens only
#         actual = get_tokens(chunk_text)
#         print(f"LLM Section {chunk_counter} | tokens = {actual}")

#         start_index = max(i - overlap_sections, start_index + 1)

#     print("\nTotal final chunks created:", chunk_counter)
#     print("Saved to:", SECTION_FILE)
#     print("===== SPLITTER END =====\n")

#     return chunk_counter

In [10]:
import re
import json
import os
from typing import List

SECTION_FILE = "../data/chunks/llm_sections.jsonl"
TOKEN_CACHE = {}

_GARBAGE_PATTERNS = [
    r"<!--\s*image\s*-->",
    r"<!-- .+? -->",
    r"^\s*Part No\.\s+\S+.*$",
    r"^\s*©.*$",
    r"^\s*\d+\s*$",
]
_GARBAGE_RE = re.compile(
    "|".join(_GARBAGE_PATTERNS),
    flags=re.IGNORECASE | re.MULTILINE
)

_HEADING_RE = re.compile(r"^(#{1,3})\s+(.+)$", re.MULTILINE)


def clean_text(text: str) -> str:
    text = _GARBAGE_RE.sub("", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def get_tokens(text: str) -> int:
    """
    Always use actual llama.cpp tokenizer.
    Cache results to avoid redundant API calls.
    Falls back to conservative estimate ONLY if server is unreachable.
    Conservative estimate: len/2.0 — intentionally pessimistic to prevent
    oversized sections from slipping through.
    """
    if not text or not text.strip():
        return 0

    key = text[:500]  # use prefix as cache key — full text is too slow to hash
    if key in TOKEN_CACHE:
        return TOKEN_CACHE[key]

    try:
        tokens = count_tokens(text)
        if tokens > 0:
            TOKEN_CACHE[key] = tokens
            return tokens
    except Exception:
        pass

    # Conservative fallback — divide by 2.0 instead of 3.8
    # Tables and sparse text can have token:char ratio as low as 1:2
    fallback = int(len(text) / 2.0)
    TOKEN_CACHE[key] = fallback
    return fallback


def parse_sections(text: str) -> List[dict]:
    """Parse H1/H2/H3 headings with breadcrumb tracking."""
    matches = list(_HEADING_RE.finditer(text))

    if not matches:
        return [{
            "level": 1,
            "heading": "Document",
            "content": text,
            "breadcrumb": "Document",
            "full_text": text
        }]

    sections = []
    h1 = h2 = ""

    for i, match in enumerate(matches):
        level = len(match.group(1))
        heading = match.group(2).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].strip()

        if level == 1:
            h1 = heading
            h2 = ""
        elif level == 2:
            h2 = heading

        if level == 1:
            breadcrumb = h1
        elif level == 2:
            breadcrumb = f"{h1} > {h2}" if h1 else h2
        else:
            h3 = heading
            breadcrumb = " > ".join(filter(None, [h1, h2, h3]))

        sections.append({
            "level": level,
            "heading": heading,
            "content": content,
            "breadcrumb": breadcrumb,
            "full_text": f"{'#' * level} {heading}\n\n{content}" if content else f"{'#' * level} {heading}",
        })

    return sections


def split_by_sentences(text: str, max_tokens: int) -> List[str]:
    """Last resort — split at sentence boundaries."""
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    current = ""

    for sentence in sentences:
        candidate = (current + " " + sentence).strip()
        if get_tokens(candidate) > max_tokens:
            if current:
                chunks.append(current)
            current = sentence
        else:
            current = candidate

    if current:
        chunks.append(current)

    return [c for c in chunks if c.strip()]


def split_by_paragraphs(text: str, max_tokens: int) -> List[str]:
    """
    Split at paragraph boundaries.
    Falls through to sentence splitting if a single paragraph is too large.
    No overlap here — overlap is handled at the batch level.
    """
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current_paras = []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = get_tokens(para)

        if para_tokens > max_tokens:
            if current_paras:
                chunks.append("\n\n".join(current_paras))
                current_paras = []
                current_tokens = 0
            chunks.extend(split_by_sentences(para, max_tokens))
            continue

        if current_tokens + para_tokens > max_tokens:
            if current_paras:
                chunks.append("\n\n".join(current_paras))
            current_paras = [para]
            current_tokens = para_tokens
        else:
            current_paras.append(para)
            current_tokens += para_tokens

    if current_paras:
        chunks.append("\n\n".join(current_paras))

    return [c for c in chunks if c.strip()]


def split_by_lines(text: str, max_tokens: int) -> List[str]:
    """
    Nuclear option — split line by line.
    Used for tables and other dense content where even paragraphs
    are too large. Each line becomes its own candidate.
    """
    lines = [l for l in text.split("\n") if l.strip()]
    chunks = []
    current_lines = []
    current_tokens = 0

    for line in lines:
        line_tokens = get_tokens(line)

        if current_tokens + line_tokens > max_tokens:
            if current_lines:
                chunks.append("\n".join(current_lines))
            current_lines = [line]
            current_tokens = line_tokens
        else:
            current_lines.append(line)
            current_tokens += line_tokens

    if current_lines:
        chunks.append("\n".join(current_lines))

    return [c for c in chunks if c.strip()]


def hard_split(text: str, max_tokens: int) -> List[str]:
    """
    Three-tier splitting cascade:
    1. Try paragraph boundaries first
    2. If any result is still too large, try sentence boundaries
    3. If still too large, split line by line
    
    This guarantees no batch exceeds max_tokens regardless of content type.
    Tables, lists, dense text — all handled.
    """
    # Tier 1 — paragraphs
    para_chunks = split_by_paragraphs(text, max_tokens)

    result = []
    for chunk in para_chunks:
        if get_tokens(chunk) <= max_tokens:
            result.append(chunk)
            continue

        # Tier 2 — sentences
        sent_chunks = split_by_sentences(chunk, max_tokens)
        for sc in sent_chunks:
            if get_tokens(sc) <= max_tokens:
                result.append(sc)
                continue

            # Tier 3 — lines
            result.extend(split_by_lines(sc, max_tokens))

    return [c for c in result if c.strip()]


def merge_sections_into_llm_batches(
    sections: List[dict],
    max_tokens: int,
    min_tokens: int,
) -> List[str]:
    """
    Merge small sections into batches, split large sections into sub-batches.
    
    Guarantees:
    - No batch exceeds max_tokens (hard guarantee via three-tier splitter)
    - Never merges across H1 chapter boundaries
    - Sections below min_tokens are merged with neighbors
    - Each oversized section sub-chunk gets a breadcrumb for LLM context
    """
    batches: List[str] = []
    current_parts: List[str] = []
    current_tokens = 0
    current_h1 = ""

    def flush() -> None:
        if current_parts:
            batches.append("\n\n".join(current_parts).strip())

    for sec in sections:

        # H1 boundary → flush and start new batch
        if sec["level"] == 1 and sec["heading"] != current_h1:
            if current_parts:
                flush()
                current_parts = []
                current_tokens = 0
            current_h1 = sec["heading"]

        section_text = sec["full_text"]
        section_tokens = get_tokens(section_text)

        # ── Oversized section → split with three-tier cascade ────────────────
        if section_tokens > max_tokens:
            # Flush buffer first
            if current_parts:
                flush()
                current_parts = []
                current_tokens = 0

            sub_chunks = hard_split(section_text, max_tokens)

            for j, sub in enumerate(sub_chunks):
                sub_tokens = get_tokens(sub)
                # Prepend breadcrumb only on first sub-chunk — rest are continuations
                if j == 0:
                    labeled = f"[Context: {sec['breadcrumb']}]\n\n{sub}"
                else:
                    labeled = f"[Context: {sec['breadcrumb']} (continued)]\n\n{sub}"

                # Final safety check — if breadcrumb pushed it over, split again
                if get_tokens(labeled) > max_tokens:
                    labeled = sub

                batches.append(labeled.strip())

            print(f"  Split oversized section '{sec['heading']}' "
                  f"({section_tokens} tokens) → {len(sub_chunks)} sub-batches")
            continue

        # ── Section fits → try to merge with current batch ───────────────────
        if current_tokens + section_tokens > max_tokens and current_parts:
            flush()

            # Overlap: carry last part if it's small enough
            last = current_parts[-1]
            last_tokens = get_tokens(last)
            if last_tokens < max_tokens * 0.25:
                current_parts = [last]
                current_tokens = last_tokens
            else:
                current_parts = []
                current_tokens = 0

        current_parts.append(section_text)
        current_tokens += section_tokens

    flush()
    return batches


def split_text_by_tokens(
    text: str,
    max_tokens: int = SAFE_INPUT_LIMIT,
    overlap_sections: int = 1,
) -> int:

    print("\n===== SPLITTER START =====")

    print("Cleaning extracted text...")
    text = clean_text(text)

    print("Parsing heading hierarchy (H1 / H2 / H3)...")
    sections = parse_sections(text)
    print(f"Total sections detected: {len(sections)}")

    for sec in sections:
        indent = "  " * (sec["level"] - 1)
        token_count = get_tokens(sec["full_text"])
        flag = " ⚠ OVERSIZED" if token_count > max_tokens else ""
        print(f"  {indent}{'#' * sec['level']} {sec['heading']} "
              f"({token_count} tokens){flag}")

    print(f"\nMerging sections into batches (max={max_tokens} tokens)...")
    min_tokens = max(50, max_tokens // 10)
    batches = merge_sections_into_llm_batches(sections, max_tokens, min_tokens)

    # ── Verify no batch exceeds limit ────────────────────────────────────────
    violations = [(i, get_tokens(b)) for i, b in enumerate(batches)
                  if get_tokens(b) > max_tokens]
    if violations:
        print(f"\n⚠ WARNING: {len(violations)} batches exceed token limit:")
        for i, t in violations:
            print(f"  Batch {i+1}: {t} tokens")
    else:
        print("\n✓ All batches within token limit")

    os.makedirs(os.path.dirname(SECTION_FILE), exist_ok=True)
    open(SECTION_FILE, "w").close()

    chunk_counter = 0
    for batch in batches:
        if not batch.strip():
            continue

        actual_tokens = get_tokens(batch)
        print(f"LLM Section {chunk_counter + 1} | tokens = {actual_tokens}")

        with open(SECTION_FILE, "a", encoding="utf-8") as f:
            json.dump({"section_text": batch}, f, ensure_ascii=False)
            f.write("\n")

        chunk_counter += 1

    print(f"\nTotal final sections created: {chunk_counter}")
    print(f"Saved to: {SECTION_FILE}")
    print("===== SPLITTER END =====\n")

    return chunk_counter

In [11]:
#Process Text Block
import re
import json
import time


def repair_json_chunks(raw: str) -> list | None:
    """
    Multi-stage JSON repair for LLM chunk output.
    Stage 1 — Strip markdown fences
    Stage 2 — Extract array boundaries
    Stage 3 — Fix unescaped real newlines inside string values
    Stage 4 — Fix trailing commas
    Stage 5 — Standard parse
    Stage 6 — Regex extraction fallback
    """

    text = raw.strip()

    # Stage 1 — strip markdown fences
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    # Stage 2 — extract from first [ to last ]
    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1:
        print("  [repair] No JSON array brackets found.")
        print(f"  [repair] Raw text preview: {text[:300]}")
        return None
    text = text[start: end + 1]

    # Stage 3 — fix unescaped real newlines inside JSON string values
    def fix_newlines_in_strings(s: str) -> str:
        result = []
        in_string = False
        escape_next = False
        for ch in s:
            if escape_next:
                result.append(ch)
                escape_next = False
                continue
            if ch == "\\":
                result.append(ch)
                escape_next = True
                continue
            if ch == '"':
                in_string = not in_string
                result.append(ch)
                continue
            if in_string and ch == "\n":
                result.append("\\n")
                continue
            if in_string and ch == "\r":
                continue
            if in_string and ch == "\t":
                result.append("\\t")
                continue
            result.append(ch)
        return "".join(result)

    text = fix_newlines_in_strings(text)

    # Stage 4 — remove trailing commas
    text = re.sub(r",\s*([\]}])", r"\1", text)

    # Stage 5 — standard parse
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list) and len(parsed) > 0:
            return parsed
        elif isinstance(parsed, list) and len(parsed) == 0:
            print("  [repair] Parsed successfully but got empty list.")
            return None
    except json.JSONDecodeError as e:
        print(f"  [repair] Standard parse failed: {e}")

    # Stage 6 — regex extraction fallback
    print("  [repair] Attempting regex extraction fallback...")
    pattern = re.compile(r'"text_chunk"\s*:\s*"((?:[^"\\]|\\.)*)"', re.DOTALL)
    matches = pattern.findall(text)
    if matches:
        print(f"  [repair] Regex extracted {len(matches)} chunks.")
        recovered = []
        for m in matches:
            try:
                value = json.loads(f'"{m}"')
                recovered.append({"text_chunk": value})
            except Exception:
                recovered.append({"text_chunk": m})
        return recovered if recovered else None

    print("  [repair] All repair stages failed.")
    return None


def process_text_block(
    system_prompt: str,
    text_block: str,
    user_prompt: str,
    max_retries: int = 3
) -> list | None:

    for attempt in range(max_retries):

        print(f"  Attempt {attempt + 1}/{max_retries}")
        raw_chunks: list[str] = []

        try:
            print("  Stage: Prefill started")
            start_time = time.time()
            first_token = True

            for token in stream_completion(system_prompt, text_block, user_prompt):
                if first_token:
                    print("  Stage: Generation started")
                    first_token = False
                raw_chunks.append(token)

            raw_response = "".join(raw_chunks)
            duration = round(time.time() - start_time, 2)
            print(f"  Completed in {duration}s")

            # Detect empty response
            if not raw_response.strip():
                print(f"  WARNING: Empty response from model (attempt {attempt + 1})")
                continue

            # Detect suspiciously fast response (cached empty/error)
            if duration < 1.0:
                print(f"  WARNING: Response in {duration}s is suspiciously fast — skipping")
                print(f"  Raw response: {repr(raw_response[:200])}")
                continue

            # Primary parse
            try:
                parsed = json.loads(raw_response.strip())
                if isinstance(parsed, list) and len(parsed) > 0:
                    print(f"  Parsed {len(parsed)} chunks (clean)\n")
                    return parsed
                elif isinstance(parsed, list) and len(parsed) == 0:
                    print(f"  WARNING: Model returned empty list []")
                    print(f"  Raw response preview: {raw_response[:300]}")
                    continue
            except json.JSONDecodeError:
                pass

            # Repair pipeline
            print("  Primary JSON parse failed — running repair pipeline...")
            print(f"  Raw response head: {raw_response[:500]}")
            print(f"  Raw response tail: {raw_response[-300:]}")

            repaired = repair_json_chunks(raw_response)

            if repaired and len(repaired) > 0:
                print(f"  Repaired successfully — {len(repaired)} chunks\n")
                return repaired

            print(f"  Repair failed on attempt {attempt + 1}")
            if attempt < max_retries - 1:
                print("  Retrying...\n")

        except Exception as e:
            print(f"  Unexpected error: {e}")
            if attempt == max_retries - 1:
                return None
            print("  Retrying...\n")

    return None

In [12]:
#Save chunks to disc
import json
import os

CHUNK_OUTPUT_FILE = "../data/chunks/chunked_dataset.jsonl"


def save_chunks_to_disk(chunks):

    os.makedirs(os.path.dirname(CHUNK_OUTPUT_FILE), exist_ok=True)

    #batch write (faster than multiple json.dump calls)
    lines = []

    for chunk in chunks:
        lines.append(json.dumps(chunk, ensure_ascii=False))

    #single write operation (much faster for large data)
    with open(CHUNK_OUTPUT_FILE, "a", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

In [13]:
#Splitter Call
import os

# Clear all previous run artifacts
for f in [
    "../data/chunks/llm_sections.jsonl",
    "../data/chunks/progress.txt",
    "../data/chunks/failed_sections.jsonl",
    "../data/chunks/chunked_dataset.jsonl",
]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted: {f}")

# Re-split with new smaller limit
print("\nRe-splitting document...\n")
total_sections = split_text_by_tokens(extracted_text)
print(f"\nNew total sections: {total_sections}")

Deleted: ../data/chunks/llm_sections.jsonl
Deleted: ../data/chunks/progress.txt
Deleted: ../data/chunks/failed_sections.jsonl
Deleted: ../data/chunks/chunked_dataset.jsonl

Re-splitting document...


===== SPLITTER START =====
Cleaning extracted text...
Parsing heading hierarchy (H1 / H2 / H3)...
Total sections detected: 4861
    ## N E X A (5 tokens)
    ## GRAND VITARA (4 tokens)
    ## OWNER'S MANUAL (160 tokens)
    ## FOREWORD (426 tokens)
    ## IMPORTANT (2 tokens)
    ## WARNING/ CAUTION/ NOTICE/ (51 tokens)
    ## WARNING (17 tokens)
    ## CAUTION (18 tokens)
    ## NOTICE (15 tokens)
    ## NOTE: (49 tokens)
    ## NOTE: (51 tokens)
    ## NOTICE (60 tokens)
    ## NOTICE (42 tokens)
    ## MODIFICATION WARNING (4 tokens)
    ## WARNING ! (45 tokens)
    ## \"WARNING\" (4 tokens)
    ## Vehicle may break-down, meet with an accident or catch fire due to (227 tokens)
    ## \"CAUTION\" (569 tokens)
    ## To experience and use the Digital Owner's Manual, download Maruti Suzuki

In [14]:
#Last Processed
import os

PROGRESS_FILE = "../data/chunks/progress.txt"


def get_last_processed_index():
    if not os.path.exists(PROGRESS_FILE):
        return -1

    with open(PROGRESS_FILE, "r") as f:
        return int(f.read().strip())


def update_progress(index):
    with open(PROGRESS_FILE, "w") as f:
        f.write(str(index))

In [15]:
#Chunking Pipeline
import json
import time
from pathlib import Path


# ── Failed section log — lets you inspect and reprocess failures later ───────
FAILED_SECTIONS_FILE = "../data/chunks/failed_sections.jsonl"


def log_failed_section(index: int, section_text: str, reason: str) -> None:
    """
    Persist failed sections to disk so they can be inspected or retried
    without re-running the entire pipeline.
    """
    Path(FAILED_SECTIONS_FILE).parent.mkdir(parents=True, exist_ok=True)
    with open(FAILED_SECTIONS_FILE, "a", encoding="utf-8") as f:
        json.dump({
            "section_index": index,
            "reason": reason,
            "section_text": section_text
        }, f, ensure_ascii=False)
        f.write("\n")


# ── Core section processor ────────────────────────────────────────────────────

def process_section(
    index: int,
    section_text: str,
    total_sections: int,
    section_retry_attempts: int = 2,
) -> bool:
    """
    Process a single section through the LLM chunker and save results.

    Returns True if the section was successfully processed and saved.
    Returns False if all attempts failed (section is logged to failed file).

    Args:
        index:                  0-based section index
        section_text:           raw text of the section
        total_sections:         total count (for progress display only)
        section_retry_attempts: how many times to re-call process_text_block
                                on top of its own internal retries
    """

    print(f"\n{'─' * 50}")
    print(f"  Section {index + 1} / {total_sections}")
    print(f"  Characters : {len(section_text)}")
    print(f"  Est. tokens: {len(section_text) // 4}")
    print(f"{'─' * 50}")

    last_error = "unknown"

    for attempt in range(1, section_retry_attempts + 1):

        if attempt > 1:
            # Exponential backoff between section-level retries
            wait = 2 ** attempt
            print(f"  [retry {attempt}/{section_retry_attempts}] Waiting {wait}s...")
            time.sleep(wait)

        try:
            result = process_text_block(system_prompt, section_text, user_prompt)

        except Exception as e:
            last_error = f"process_text_block raised: {e}"
            print(f"  ERROR: {last_error}")
            continue

        # ── Validate result before saving ────────────────────────────────────
        if result is None:
            last_error = "process_text_block returned None (all internal retries exhausted)"
            print(f"  WARNING: {last_error}")
            continue

        if not isinstance(result, list) or len(result) == 0:
            last_error = f"Unexpected result type or empty list: {type(result)}"
            print(f"  WARNING: {last_error}")
            continue

        # ── Validate chunk content ────────────────────────────────────────────
        valid_chunks = []
        for chunk in result:
            if not isinstance(chunk, dict):
                continue
            text = chunk.get("text_chunk", "").strip()
            if len(text) < 20:   # skip empty / near-empty chunks
                print(f"  Skipping suspiciously short chunk ({len(text)} chars)")
                continue
            valid_chunks.append({"text_chunk": text})

        if not valid_chunks:
            last_error = "All chunks failed validation (empty or too short)"
            print(f"  WARNING: {last_error}")
            continue

        # ── Save — update progress only after confirmed save ──────────────────
        try:
            save_chunks_to_disk(valid_chunks)
        except Exception as e:
            last_error = f"save_chunks_to_disk failed: {e}"
            print(f"  ERROR: {last_error}")
            continue

        # Progress is written AFTER save succeeds
        update_progress(index)

        print(f"  ✓ Saved {len(valid_chunks)} chunks "
              f"({len(result) - len(valid_chunks)} skipped)")
        return True

    # ── All attempts failed ───────────────────────────────────────────────────
    print(f"  ✗ Section {index + 1} FAILED after {section_retry_attempts} attempts.")
    print(f"  Reason: {last_error}")
    print(f"  Logging to: {FAILED_SECTIONS_FILE}")
    log_failed_section(index, section_text, last_error)
    return False


# ── Pipeline runner ───────────────────────────────────────────────────────────

def run_chunking_parallel_resume(
    section_file: str = SECTION_FILE,
    section_retry_attempts: int = 2,
) -> None:
    """
    Resume-safe sequential pipeline runner.

    - Reads sections from JSONL (one section per line)
    - Skips already-processed sections using progress tracker
    - Logs failures without stopping the pipeline
    - Prints a summary when done
    """

    # ── Count total sections first (needed for progress display) ──────────────
    section_file_path = Path(section_file)
    if not section_file_path.exists():
        print(f"ERROR: Section file not found: {section_file}")
        return

    with open(section_file_path, "r", encoding="utf-8") as f:
        total_sections = sum(1 for line in f if line.strip())

    if total_sections == 0:
        print("ERROR: Section file is empty. Run split_text_by_tokens() first.")
        return

    # ── Resume point ──────────────────────────────────────────────────────────
    last_done = get_last_processed_index()
    start_from = last_done + 1

    if start_from >= total_sections:
        print("All sections already processed. Nothing to do.")
        print("Delete progress.txt to reprocess from scratch.")
        return

    print(f"\n{'=' * 50}")
    print(f"  CHUNKING PIPELINE")
    print(f"  Total sections : {total_sections}")
    print(f"  Already done   : {start_from}")
    print(f"  Remaining      : {total_sections - start_from}")
    print(f"{'=' * 50}")

    # ── Main loop ─────────────────────────────────────────────────────────────
    succeeded = 0
    failed = 0
    pipeline_start = time.time()

    with open(section_file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):

            # Skip already-processed sections
            if i < start_from:
                continue

            line = line.strip()
            if not line:
                continue

            try:
                section_text = json.loads(line)["section_text"]
            except (json.JSONDecodeError, KeyError) as e:
                print(f"\n  ERROR reading section {i + 1}: {e}")
                print(f"  Raw line: {line[:100]}...")
                failed += 1
                log_failed_section(i, line, f"JSONL parse error: {e}")
                continue

            success = process_section(
                index=i,
                section_text=section_text,
                total_sections=total_sections,
                section_retry_attempts=section_retry_attempts,
            )

            if success:
                succeeded += 1
            else:
                failed += 1

    # ── Summary ───────────────────────────────────────────────────────────────
    total_time = round(time.time() - pipeline_start, 1)
    total_processed = succeeded + failed

    print(f"\n{'=' * 50}")
    print(f"  CHUNKING COMPLETE")
    print(f"{'=' * 50}")
    print(f"  Processed : {total_processed} sections")
    print(f"  Succeeded : {succeeded}")
    print(f"  Failed    : {failed}")
    print(f"  Time      : {total_time}s")

    if failed > 0:
        print(f"\n  {failed} section(s) failed.")
        print(f"  Failed sections saved to: {FAILED_SECTIONS_FILE}")
        print(f"  You can inspect and reprocess them separately.")

    print(f"{'=' * 50}\n")

In [ ]:
run_chunking_parallel_resume()


  CHUNKING PIPELINE
  Total sections : 252
  Already done   : 0
  Remaining      : 252

──────────────────────────────────────────────────
  Section 1 / 252
  Characters : 11105
  Est. tokens: 2776
──────────────────────────────────────────────────
  Attempt 1/3
  Stage: Prefill started
  Stage: Generation started
  Completed in 72.68s
  Parsed 18 chunks (clean)

  ✓ Saved 18 chunks (0 skipped)

──────────────────────────────────────────────────
  Section 2 / 252
  Characters : 8332
  Est. tokens: 2083
──────────────────────────────────────────────────
  Attempt 1/3
  Stage: Prefill started
  Stage: Generation started
  Completed in 73.33s
  Parsed 13 chunks (clean)

  Skipping suspiciously short chunk (9 chars)
  ✓ Saved 12 chunks (1 skipped)

──────────────────────────────────────────────────
  Section 3 / 252
  Characters : 5275
  Est. tokens: 1318
──────────────────────────────────────────────────
  Attempt 1/3
  Stage: Prefill started
  Stage: Generation started
  Completed in 90

In [ ]:
# #Chunking
# with open(SECTION_FILE, "r", encoding="utf-8") as f:

#         for i, line in enumerate(f):
# #
#             section = json.loads(line)["section_text"]

#             print("\n----------------------------------")
#             print(f"Processing section {i+1}/{total_sections}")
#             print("----------------------------------")

#             section_output = process_text_block(system_prompt, section, user_prompt)

#             if section_output:

#                 save_chunks_to_disk(section_output)

#                 print(f"Saved {len(section_output)} chunks to disk.")

#                 del section_output

#             del section


# print("\n==============================")
# print(" CHUNKING COMPLETE")
# print("==============================")


----------------------------------
Processing section 1/74
----------------------------------
Stage: CPU → Prefill started
Stage: GPU → Generation started
An error occurred: ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Completed in 16.36s

JSON parsing failed (attempt 1/3)
JSON repair failed

Retrying...

Stage: CPU → Prefill started
An error occurred: HTTPConnectionPool(host='127.0.0.1', port=36912): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=36912): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
Completed in 2.05s

JSON parsing failed (attempt 2/3)
JSON repair failed

Retrying...

Stage: CPU → Prefill started
An error occu

KeyboardInterrupt: 

In [ ]:
# #Automatic Pipeline
# import json

# def run_chunking_pipeline():

#     print(" STARTING CHUNKING PIPELINE\n")

#     print("Document token count:", document_tokens)
#     print("Model context limit :", MAX_CONTEXT_SIZE)

#     if total_tokens <= MAX_CONTEXT_SIZE:

#         print("\nDocument fits inside context window.")
#         print("Processing entire document...\n")

#         result = process_text_block(system_prompt, extracted_text, user_prompt)

#         if result:
#             save_chunks_to_disk(result)

#         print("Chunks written to disk.")
#         return


#     print("\nDocument exceeds context window.")
#     print("Splitting document into sections...\n")

#     total_sections = split_text_by_tokens(extracted_text)

#     print("Total sections:", total_sections)


#     with open(SECTION_FILE, "r", encoding="utf-8") as f:

#         for i, line in enumerate(f):

#             section = json.loads(line)["section_text"]

#             print("\n----------------------------------")
#             print(f"Processing section {i+1}/{total_sections}")
#             print("----------------------------------")

#             section_tokens = count_tokens(section)
#             print("Section token count:", section_tokens)

#             section_output = process_text_block(system_prompt, section, user_prompt)

#             if section_output:

#                 save_chunks_to_disk(section_output)

#                 print(f"Saved {len(section_output)} chunks to disk.")

#                 del section_output

#             del section


#     print("\n==============================")
#     print(" CHUNKING COMPLETE")
#     print("==============================")

In [ ]:
# full_response = run_chunking_pipeline()

 STARTING CHUNKING PIPELINE

Document token count: 544243
Model context limit : 32768

Document exceeds context window.
Splitting document into sections...


===== SPLITTER START =====
Detecting markdown sections...
Total sections detected: 4866

Combining sections while respecting token limit...

Total final chunks created: 25
Saved to: ../data/chunks/llm_sections.jsonl
===== SPLITTER END =====

Total sections: 25

----------------------------------
Processing section 1/25
----------------------------------
Section token count: 23953


In [ ]:
# import os

# os.makedirs("../data/chunks", exist_ok=True)

# with open("../data/chunks/grand_vitara_semantic_chunks.json", "w", encoding="utf-8") as f:
#     f.write(full_response)

# print("Chunks saved successfully.")